# **TP4 | Inverse multiplicatif modulaire**

## 4.1. Coefficients de Bézout

### 4.1.a. Algorithme d'Euclide

Calculer tous les diviseurs d'un nombre peut être très long, de complexité $\mathcal{O}(n^{1/2})$, et est infaisable en pratique pour des nombres de 2048 bits. 
Pour calculer le PGCD, on préfère donc utiliser la propriété, pour $x \in \mathbb{Z}$ et $y \in \mathbb{Z} \setminus \{0\}$,

$\qquad\qquad\mathrm{pgcd}(x, y) = \mathrm{pgcd}(y, x \% y) . $

Cette propriété est à la base de l'algorithme d'Euclide, qui calcule le PGCD de deux entiers en $\mathcal{O}(\log |y|)$.

Pour $y = 0$, on a $\mathrm{pgcd}(x, 0) = |x|$.

**TODO**\
Traduire l'algorithme d'Euclide en code. Le voici en pseudo-code :

> Entrées : 𝑥 ∈ ℤ et 𝑦 ∈ ℤ\
> Sortie : pgcd(𝑥, 𝑦)\
> Tant que 𝑦 ≠ 0\
> -|--- 𝑥, 𝑦 ← 𝑦, 𝑥%𝑦\
> Renvoyer |𝑥|

Tester cet algorithme sur les couples (46, 23), (21, 20) et (33, 42).

In [1]:
def pgcd(x: int, y: int):
    """Calcule le PGCD de x,y avec l'algorithme d'Euclide."""
    while y != 0:
        x, y = y, x % y
    return abs(x)

# D(46) = {1,2,23,46} et D(23) = {1,23}
print("pgcd(46, 23) = 23:", pgcd(46, 23) == 23)
# D(21) = {1,3,7,21} et D(20) = {1,2,4,5,10,20}
print("pgcd(21, 20) = 1:", pgcd(21, 20) == 1)
# D(33) = {1,3,11,33} et D(42) = {1,2,3,6,7}
print("pgcd(33, 42) = 3:", pgcd(33, 42) == 3)

pgcd(46, 23) = 23: True
pgcd(21, 20) = 1: True
pgcd(33, 42) = 3: True


### 4.1.b. Algorithme d'Euclide étendu

On cherche des coefficients de Bézout $u,v$ tels que

$\qquad\qquad u\times x + v\times y = \mathrm{pgcd}(x, y). $

Si $y = 0$ et $x \geq 0$, alors on a $u = 1$ et $v = 0$. Si $y = 0$ et $x < 0$, alors on a $u = -1$ et $v = 0$.

Sinon, on peut écrire $x = q\times y + r$ avec $0 \leq r < |y|$. On a alors $\mathrm{pgcd}(x, y) = \mathrm{pgcd}(y, r)$.
Si $u',v'$ sont des coefficients de Bézout pour $y$ et $r$, on a

$\qquad\qquad u'\times y + v'\times r = \mathrm{pgcd}(y, r) = \mathrm{pgcd}(x, y) $

d'où l'identité 

$\qquad\qquad\begin{aligned}
  \mathrm{pgcd}(x, y) & = u'\times y + v'\times (x - q\times y) \\
  & = v'\times x + (u' - v'\times q)\times y
\end{aligned}$

et donc $u = v'$ et $v = u' - v'\times q$ sont des coefficients de Bézout pour $x$ et $y$.

*Remarque :* Ces coefficients ne sont pas uniques !

In [2]:
def bezout(x: int, y: int):
    """
    Calcule de manière récursive des coefficients de Bézout pour x et y :
    deux entiers u et v tels que u * x + v * y = pgcd(x, y).
    """
    if y == 0:
        u = 1 if x >= 0 else -1
        v = 0
        return u, v

    q, r = divmod(x, y)
    up, vp = bezout(y, r)
    return vp, up - vp * q

### 4.1.c. Validation du code

**TODO:**  
Tester la fonction ci-dessus en vérifiant qu'on obtient bien des identités de Bézout pour les couples (46, 21), (214, 76) et (417, 55).

In [3]:
def aux_test_pgcd(x, y):
    d = pgcd(x, y)
    u, v = bezout(x, y)
    return u * x + v * y == d

x1, y1 = 46, 21
valid1 = aux_test_pgcd(x1, y1)
x2, y2 = 214, 76
valid2 = aux_test_pgcd(x2, y2)
x3, y3 = 417, 55
valid3 = aux_test_pgcd(x3, y3)

print(f"On a une identité de Bézout pour le couple {x1, y1}:", valid1)
print(f"On a une identité de Bézout pour le couple {x2, y2}:", valid2)
print(f"On a une identité de Bézout pour le couple {x3, y3}:", valid3)

On a une identité de Bézout pour le couple (46, 21): True
On a une identité de Bézout pour le couple (214, 76): True
On a une identité de Bézout pour le couple (417, 55): True


## 4.2. Calcul d'inverse

Si $\mathrm{pgcd}(x, n) = 1$, on peut trouver l'inverse multiplicatif de $x$ modulo $n$. En effet, 

$\qquad\qquad \begin{aligned}
& u\times x + v \times n = 1 \\ \Rightarrow\quad & u\times x \equiv 1\ \mod n \\ \Rightarrow\quad & \mathrm{inv}_n(x) = u \% n \end{aligned} $

Réciproquement, si on trouve $u \in \mathbb{Z}$ tel que $u\times x \equiv 1 \mod n$, alors $\mathrm{pgcd}(x, y) = 1$.

### 4.2.a. Fonction `inv_mod`

**TODO:**\
Coder une fonction `inv_mod(x, n)` qui calcule l'inverse de `x` modulo `n` lorsque cela est possible, en utilisant la fonction `bezout` précédente. Si l'inverse n'existe pas, la fonction devra renvoyer une erreur.

In [4]:
def inv_mod(x, n):
    """
    Calcule l'inverse de x modulo n. Si l'inverse n'existe pas,
    renvoyer une erreur.

    Args:
        x (int): Le nombre à inverser
        n (int): Le modulus

    Returns:
        int: L'inverse de x dans Z / nZ

    Raises:
        ValueError: Si le PGCD de x et n n'est pas 1.
    """
    u, v = bezout(x, n)

    pgcd = u * x + v * n
    if pgcd != 1:
        raise ValueError(f"{x} n'est pas inversible modulo {n}")

    return u % n

### 4.2.b. Validation du code

**TODO:**  
Tester la fonction sur les exemples 
- $x = 21, n = 46$
- $x = 20, n = 21$
- $x = 33, n = 42$

Comparer avec les résultats de la fonction `pow(x, -1, n)` de Python.
Vérifier que cela ne fonctionne pas sur un exemple $\mathrm{pgcd}(x, n) \neq 1$.

In [5]:
inv_mod(21, 46) == pow(21, -1, 46)

True

## 4.3. Bonus: Pour aller plus loin

### 4.3.a. Chiffrement affine

Considérons un chiffrement un peu meilleur que César sur l'alphabet $\mathcal{A} = \{0, 1, ..., 25\}$, défini par

$\qquad\qquad c = (a\times m + b) \% 26 $

Par exemple avec les paramètres $a = 5$ et $b = 8$, on a

| lettre claire | a | b | c | d | e | f | g | h | i | j | k | l | m | n | o | p | q | r | s | t | u | v | w | x | y | z |
|---|---|---|---|---|---|---|---|---|---|---|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|----|
| nombre clair $m$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 | 20 | 21 | 22 | 23 | 24 | 25 |
| nombre chiffré $c$ | 8 | 13 | 18 | 23 | 2 | 7 | 12 | 17 | 22 | 1 | 6 | 11 | 16 | 21 | 0 | 5 | 10 | 15 | 20 | 25 | 4 | 9 | 14 | 19 | 24 | 3 |
| lettre chiffrée | i | n | s | x | c | h | m | r | w | b | g | l | q | v | a | f | k | p | u | z | e | j | o | t | y | d |

On vérifie cela ci-dessous.

In [6]:
from affine_cesar import AffineCesarParams, encrypt

params = AffineCesarParams(5, 8)
encrypt("abcdefghijklmnopqrstuvwxyz", params)

'insxchmrwbglqvafkpuzejotyd'

**TODO**\
Compléter la classe `InverseAffineCesarParams` ci-dessous avec $a'$ et $b'$ tels que

$\qquad\qquad m = (a'\times c + b') \% 26 $

Vérifier sur `"insxchmrwbglqvafkpuzejotyd"` qu'on retrouve bien l'alphabet initial en chiffrant avec ces nouveaux paramètres.

In [7]:
class InverseAffineCesarParams:
    def __init__(self, params: AffineCesarParams):
        # Modulo 26, on a par définition c = a * m + b
        # donc a * m = c - b
        # et donc m = inv(a) * (c - b) = inv(a) * c - inv(a) * b
        self.a = pow(params.a, -1, 26)
        self.b = (self.a * (-params.b)) % 26


inv_params = InverseAffineCesarParams(params)
encrypt("insxchmrwbglqvafkpuzejotyd", inv_params)

'abcdefghijklmnopqrstuvwxyz'

### 4.3.b. Complexité du calcul de diviseurs

**TODO**\
Générer deux nombres aléatoires de 128 bits, et comparer les temps de calcul de :
- la fonction `galois.divisors` de la librairie `galois` (qui calcule tous les diviseurs d'un nombre)
- la fonction `pgcd` définie ci-dessus (qui calcule le PGCD de deux nombres)

On pourra utiliser la commande magique `%time <code>` pour mesurer le temps d'exécution d'une ligne de code.

In [8]:
import os
import galois

x = int.from_bytes(os.urandom(16), "big")
y = int.from_bytes(os.urandom(16), "big")

In [9]:
%time galois.divisors(x), galois.divisors(y);

CPU times: user 3.06 s, sys: 3.12 ms, total: 3.06 s
Wall time: 3.07 s


In [10]:
%time pgcd(x, y);

CPU times: user 12 μs, sys: 1 μs, total: 13 μs
Wall time: 14.1 μs


In [11]:
# La différence de temps de calcul est énorme (même si elle est aléatoire en fonction des tirages de x et y)
# par exemple, calcul de diviseurs 932 ms et pgcd 10.3 µs
# ce qui correspond à un facteur presque 100 000.